# $\pi^0$ Dalitz decays ($\pi^0 \to e^+ e^- \gamma$) in the Geant4 BNB overlay

This notebook loads the BNB `nu_overlay` and pi0-enriched `nc_pi0_overlay` checkout files, finds truth $\pi^0$ Dalitz decays, and extracts the $\pi^0$, $\gamma$, $e^+$, and $e^-$ four-momenta. It then makes a series of kinematic plots, both in the lab frame and in the $\pi^0$ / $e^+e^-$ rest frames.

The goal is to characterize the Geant4 treatment of this 3-body decay so it can later be compared against a dedicated EvtGen simulation (Geant4 is suspected of doing the Dalitz decay less accurately than the full QED matrix element).

**Dalitz selection (per the convention in `src/postprocessing.py`):** a truth $\pi^0$ (`truth_pdg == 111`) whose set of direct daughters (matched via `truth_mother == pi0 truth_id`) contains a $\gamma$ (22), an $e^-$ (11), and an $e^+$ (-11). Any such $\pi^0$ is treated as a Dalitz decay.

In [ ]:
import os
import numpy as np
import uproot
import matplotlib.pyplot as plt

# ---- physical constants [MeV] ----
M_PI0 = 134.9768
M_E = 0.51099895

DATA_DIR = "/nevis/riverside/data/leehagaman/ngem/data_files/"

# (sample label, filename) for every checkout file to extract Dalitz decays from.
# The NC pi0 overlay files are pi0-enriched, so they dominate the Dalitz statistics.
INPUT_FILES = [
    ("nu_overlay",     "checkout_MCC9.10_Run123_v10_04_07_20_BNB_nu_overlay_surprise_reco2_hist_1.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4a_v10_04_07_16_BNB_NCpi0_overlay_surprise_reco2_hist.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4b_v10_04_07_09_BNB_NC_pi0_overlay_surprise_reco2_hist.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4c4d5_v10_04_07_13_BNB_NCpi0_overlay_surprise_reco2_hist_4c.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4c4d5_v10_04_07_13_BNB_NCpi0_overlay_surprise_reco2_hist_4d.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4c4d5_v10_04_07_13_BNB_NCpi0_overlay_surprise_reco2_hist_5.root"),
]

PLOT_DIR = "../plots/pi0_dalitz"
os.makedirs(PLOT_DIR, exist_ok=True)


def savefig(name):
    plt.savefig(os.path.join(PLOT_DIR, name + ".pdf"))
    plt.savefig(os.path.join(PLOT_DIR, name + ".png"), dpi=200)

## Lorentz-transformation helpers

Four-vectors are stored as `[E, px, py, pz]` (MeV). `boost_to_rest` boosts a set of four-vectors into the rest frame of a given "frame" four-vector (e.g. the $\pi^0$ or the $e^+e^-$ system).

In [ ]:
def boost_to_rest(p4, frame_p4):
    """Boost four-vectors `p4` (..., 4) into the rest frame of `frame_p4` (..., 4).

    Both are [E, px, py, pz]. Returns boosted four-vectors with the same shape as p4.
    """
    p4 = np.asarray(p4, dtype=float)
    frame_p4 = np.asarray(frame_p4, dtype=float)
    E_frame = frame_p4[..., 0]
    beta = frame_p4[..., 1:] / E_frame[..., None]      # frame velocity in the lab
    b2 = np.sum(beta**2, axis=-1)
    gamma = 1.0 / np.sqrt(1.0 - b2)
    pe = p4[..., 0]
    pv = p4[..., 1:]
    bp = np.sum(beta * pv, axis=-1)                    # beta . p
    gamma2 = np.where(b2 > 0, (gamma - 1.0) / b2, 0.0)
    E_prime = gamma * (pe - bp)
    pv_prime = pv + (gamma2 * bp - gamma * pe)[..., None] * beta
    return np.concatenate([E_prime[..., None], pv_prime], axis=-1)


def p3(p4):
    """Return the 3-momentum part of a four-vector."""
    return p4[..., 1:]


def pmag(p4):
    """|p| of a four-vector."""
    return np.linalg.norm(p4[..., 1:], axis=-1)


def angle_between(a3, b3):
    """Angle [deg] between two 3-vectors (arrays of shape (..., 3))."""
    cos = np.sum(a3 * b3, axis=-1) / (
        np.linalg.norm(a3, axis=-1) * np.linalg.norm(b3, axis=-1)
    )
    cos = np.clip(cos, -1.0, 1.0)
    return np.degrees(np.arccos(cos))


# ---- sanity check: boosting a particle into its own rest frame ----
_test = np.array([[M_PI0 * 1.5, 50.0, -30.0, 80.0]])  # arbitrary moving pi0-mass particle
_rest = boost_to_rest(_test, _test)
print("rest-frame E (should be ~mass):", _rest[0, 0])
print("rest-frame |p| (should be ~0):", pmag(_rest)[0])

## Load truth particle lists and select Dalitz decays

We read the WireCell `T_PFeval` truth particle arrays from every file in `INPUT_FILES` (the Run 1 `nu_overlay` file plus the pi0-enriched `nc_pi0_overlay` files, for much higher Dalitz statistics). `truth_startMomentum` is `[px, py, pz, E]` in GeV; we convert to MeV. While looping we also count **all** truth $\pi^0$s so we can verify the Dalitz fraction against the known branching ratio (BR$(\pi^0\to e^+e^-\gamma) \approx 1.174\%$). Each extracted decay keeps its sample label (`nu_overlay` / `nc_pi0_overlay`).

In [ ]:
BRANCHES = ["run", "subrun", "event", "truth_id", "truth_pdg", "truth_mother", "truth_startMomentum"]


def extract_dalitz(filepath):
    """Extract pi0 Dalitz decays from one checkout file.

    Returns (rows, n_pi0_total, n_pi0_dalitz) where rows is a dict of per-decay
    lab-frame four-vectors [E, px, py, pz] (MeV) plus run/subrun/event. Reads in
    chunks to keep memory bounded for the large NC pi0 files.
    """
    rows = {k: [] for k in ["pi0", "gamma", "ep", "em", "run", "subrun", "event"]}
    n_pi0_total = 0
    n_pi0_dalitz = 0
    for chunk in uproot.iterate(f"{filepath}:wcpselection/T_PFeval", BRANCHES,
                                step_size="500 MB", library="np"):
        n = len(chunk["truth_pdg"])
        for i in range(n):
            pdg = np.asarray(chunk["truth_pdg"][i])
            if pdg.size == 0:
                continue
            tid = np.asarray(chunk["truth_id"][i])
            moth = np.asarray(chunk["truth_mother"][i])
            sm = np.asarray(chunk["truth_startMomentum"][i], dtype=float)  # [px,py,pz,E] GeV

            for k in np.where(pdg == 111)[0]:
                n_pi0_total += 1
                daughter_mask = moth == tid[k]
                dpdg = pdg[daughter_mask]
                if not (22 in dpdg and 11 in dpdg and -11 in dpdg):
                    continue
                d_idx = np.where(daughter_mask)[0]
                g_i = d_idx[pdg[d_idx] == 22][0]
                em_i = d_idx[pdg[d_idx] == 11][0]
                ep_i = d_idx[pdg[d_idx] == -11][0]
                n_pi0_dalitz += 1

                def p4_mev(idx):  # [px,py,pz,E] GeV -> [E,px,py,pz] MeV
                    px, py, pz, E = sm[idx]
                    return [E * 1000.0, px * 1000.0, py * 1000.0, pz * 1000.0]

                rows["pi0"].append(p4_mev(k))
                rows["gamma"].append(p4_mev(g_i))
                rows["ep"].append(p4_mev(ep_i))
                rows["em"].append(p4_mev(em_i))
                rows["run"].append(int(chunk["run"][i]))
                rows["subrun"].append(int(chunk["subrun"][i]))
                rows["event"].append(int(chunk["event"][i]))
    return rows, n_pi0_total, n_pi0_dalitz


# accumulate Dalitz decays across all input files
pi0_p4, gamma_p4, ep_p4, em_p4 = [], [], [], []
dalitz_run, dalitz_subrun, dalitz_event, dalitz_sample = [], [], [], []
n_pi0_total = 0
n_pi0_dalitz = 0

print(f"{'sample':<16} {'pi0s':>10} {'Dalitz':>8} {'frac %':>8}  file")
for label, fname in INPUT_FILES:
    rows, npi0, ndal = extract_dalitz(os.path.join(DATA_DIR, fname))
    pi0_p4 += rows["pi0"]; gamma_p4 += rows["gamma"]; ep_p4 += rows["ep"]; em_p4 += rows["em"]
    dalitz_run += rows["run"]; dalitz_subrun += rows["subrun"]; dalitz_event += rows["event"]
    dalitz_sample += [label] * len(rows["pi0"])
    n_pi0_total += npi0
    n_pi0_dalitz += ndal
    fr = ndal / npi0 * 100 if npi0 else 0.0
    print(f"{label:<16} {npi0:>10,} {ndal:>8,} {fr:>8.3f}  {fname}")

pi0_p4 = np.array(pi0_p4)
gamma_p4 = np.array(gamma_p4)
ep_p4 = np.array(ep_p4)
em_p4 = np.array(em_p4)
dalitz_sample = np.array(dalitz_sample)

print("-" * 70)
print(f"TOTAL truth pi0s: {n_pi0_total:,}   Dalitz pi0s: {n_pi0_dalitz:,}   "
      f"fraction: {n_pi0_dalitz / n_pi0_total * 100:.3f}%  (PDG BR = 1.174%)")

In [ ]:
# quick four-momentum conservation / invariant-mass sanity checks (lab frame)
sum_daughters = gamma_p4 + ep_p4 + em_p4
print("max |E(pi0) - sum E(daughters)| [MeV]:", np.max(np.abs(pi0_p4[:, 0] - sum_daughters[:, 0])))

pi0_mass = np.sqrt(np.clip(pi0_p4[:, 0]**2 - pmag(pi0_p4)**2, 0, None))
print(f"reconstructed pi0 invariant mass: {pi0_mass.mean():.3f} +/- {pi0_mass.std():.3f} MeV (expect {M_PI0})")

## Save the extracted four-momenta

Saved as an `.npz` (each array is `(N, 4)` = `[E, px, py, pz]` in MeV) for downstream comparison with EvtGen.

In [ ]:
OUT_NPZ = "pi0_dalitz_momenta.npz"
np.savez(
    OUT_NPZ,
    pi0_p4=pi0_p4,
    gamma_p4=gamma_p4,
    ep_p4=ep_p4,
    em_p4=em_p4,
    sample=dalitz_sample,
    run=np.array(dalitz_run),
    subrun=np.array(dalitz_subrun),
    event=np.array(dalitz_event),
)
print(f"saved {len(pi0_p4):,} Dalitz decays to {OUT_NPZ}")

## Lab-frame energy distributions

$\pi^0$, $\gamma$, $e^+$, and $e^-$ total energies in the lab frame.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].hist(pi0_p4[:, 0], bins=50, histtype="step", color="k")
axes[0, 0].set_xlabel(r"$E_{\pi^0}$ [MeV]")

axes[0, 1].hist(gamma_p4[:, 0], bins=50, histtype="step", color="tab:orange")
axes[0, 1].set_xlabel(r"$E_{\gamma}$ [MeV]")

axes[1, 0].hist(ep_p4[:, 0], bins=50, histtype="step", color="tab:red")
axes[1, 0].set_xlabel(r"$E_{e^+}$ [MeV]")

axes[1, 1].hist(em_p4[:, 0], bins=50, histtype="step", color="tab:blue")
axes[1, 1].set_xlabel(r"$E_{e^-}$ [MeV]")

for ax in axes.flat:
    ax.set_ylabel("Dalitz decays")
fig.suptitle("Lab-frame energies")
fig.tight_layout()
savefig("lab_frame_energies")
plt.show()

## Lab-frame $e^+e^-$ pair kinematics

Opening angle, total energy, and energy asymmetry of the $e^+e^-$ pair, in the **lab frame**. The opening angle is strongly collimated here because the pair is boosted along the $\pi^0$ flight direction.

In [ ]:
ee_p4_lab = ep_p4 + em_p4
ee_energy_lab = ee_p4_lab[:, 0]                       # E_e+ + E_e-
ee_opening_lab = angle_between(p3(ep_p4), p3(em_p4))  # opening angle, lab
ee_asym_lab = (ep_p4[:, 0] - em_p4[:, 0]) / (ep_p4[:, 0] + em_p4[:, 0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(ee_opening_lab, bins=50, histtype="step", color="tab:purple")
axes[0].set_xlabel(r"$e^+e^-$ opening angle [deg] (lab)")
axes[1].hist(ee_energy_lab, bins=50, histtype="step", color="tab:cyan")
axes[1].set_xlabel(r"$E_{e^+} + E_{e^-}$ [MeV] (lab)")
axes[2].hist(ee_asym_lab, bins=np.linspace(-1, 1, 51), histtype="step", color="tab:green")
axes[2].set_xlabel(r"energy asymmetry $(E_{e^+}-E_{e^-})/(E_{e^+}+E_{e^-})$ (lab)")
for ax in axes:
    ax.set_ylabel("Dalitz decays")
fig.suptitle(r"Lab-frame $e^+e^-$ pair kinematics")
fig.tight_layout()
savefig("lab_frame_ee_kinematics")
plt.show()

## Boost to the $\pi^0$ rest frame

We boost the $\gamma$, $e^+$, and $e^-$ four-momenta into the rest frame of their parent $\pi^0$. In this frame the total energy is fixed ($E_\gamma + E_{e^+} + E_{e^-} = m_{\pi^0}$), but the individual energies vary event-to-event because this is a 3-body decay — their distributions encode the decay matrix element.

**Notation:** a star (e.g. $E^{*}_{\gamma}$, $\theta^{*}$) denotes a quantity evaluated in the $\pi^0$ rest frame, as opposed to the lab-frame quantities above.

In [ ]:
gamma_rest = boost_to_rest(gamma_p4, pi0_p4)
ep_rest = boost_to_rest(ep_p4, pi0_p4)
em_rest = boost_to_rest(em_p4, pi0_p4)

# check: energies sum to the pi0 mass in its rest frame
etot_rest = gamma_rest[:, 0] + ep_rest[:, 0] + em_rest[:, 0]
print(f"sum of rest-frame energies: {etot_rest.mean():.3f} +/- {etot_rest.std():.3f} MeV (expect {M_PI0})")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(gamma_rest[:, 0], bins=50, histtype="step", color="tab:orange")
axes[0].set_xlabel(r"$E^{*}_{\gamma}$ [MeV]")
axes[1].hist(ep_rest[:, 0], bins=50, histtype="step", color="tab:red")
axes[1].set_xlabel(r"$E^{*}_{e^+}$ [MeV]")
axes[2].hist(em_rest[:, 0], bins=50, histtype="step", color="tab:blue")
axes[2].set_xlabel(r"$E^{*}_{e^-}$ [MeV]")
for ax in axes:
    ax.set_ylabel("Dalitz decays")
fig.suptitle(r"Energies in the $\pi^0$ rest frame ($E^{*}$)")
fig.tight_layout()
savefig("pi0_rest_frame_energies")
plt.show()

## $e^+e^-$ opening angle and energy asymmetry (in the $\pi^0$ rest frame)

- **Opening angle:** angle between the $e^+$ and $e^-$ 3-momenta in the $\pi^0$ rest frame.
- **Energy asymmetry:** $y = (E^{*}_{e^+} - E^{*}_{e^-}) / (E^{*}_{e^+} + E^{*}_{e^-})$, computed in the $\pi^0$ rest frame. For an unpolarized, CP-symmetric decay this should be symmetric about 0.

In [ ]:
opening_angle = angle_between(p3(ep_rest), p3(em_rest))
asymmetry = (ep_rest[:, 0] - em_rest[:, 0]) / (ep_rest[:, 0] + em_rest[:, 0])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(opening_angle, bins=50, histtype="step", color="tab:purple")
axes[0].set_xlabel(r"$e^+e^-$ opening angle [deg] ($\pi^0$ rest frame)")
axes[0].set_ylabel("Dalitz decays")

axes[1].hist(asymmetry, bins=np.linspace(-1, 1, 51), histtype="step", color="tab:green")
axes[1].set_xlabel(r"energy asymmetry $(E_{e^+}-E_{e^-})/(E_{e^+}+E_{e^-})$")
axes[1].set_ylabel("Dalitz decays")
fig.tight_layout()
savefig("opening_angle_and_asymmetry")
plt.show()

## $\gamma$ angle in the $e^+e^-$ rest frame

We boost into the rest frame of the $e^+e^-$ pair and measure angles relative to the $\pi^0$ flight direction (i.e. we “rotate so the $\pi^0$ momentum is along $z$” — the polar angle relative to $z$ is just the angle relative to the $\pi^0$ direction, computed here via a dot product).

**Note on the $\gamma$ angle:** by four-momentum conservation, in the $e^+e^-$ rest frame the pair's 3-momentum is zero, so $\vec{p}_{\pi^0} = \vec{p}_\gamma$ exactly. The $\gamma$ is therefore *collinear* with the $\pi^0$ by construction, and its polar angle w.r.t. $z$ is identically 0. We plot it as a sanity check, and also plot the physically meaningful **$e^+$ polar angle** — the standard Dalitz lepton angle, which carries the virtual-photon polarization information that should differ between Geant4 and EvtGen.

In [ ]:
ee_system = ep_p4 + em_p4  # e+e- pair four-momentum (lab)

pi0_in_ee = boost_to_rest(pi0_p4, ee_system)
gamma_in_ee = boost_to_rest(gamma_p4, ee_system)
ep_in_ee = boost_to_rest(ep_p4, ee_system)

z_dir = p3(pi0_in_ee)  # pi0 direction defines +z
gamma_angle = angle_between(p3(gamma_in_ee), z_dir)
ep_angle = angle_between(p3(ep_in_ee), z_dir)

print(f"gamma angle wrt pi0 direction: mean {gamma_angle.mean():.3e} deg, max {gamma_angle.max():.3e} deg (expect ~0)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(gamma_angle, bins=50, histtype="step", color="tab:orange")
axes[0].set_xlabel(r"angle of $\gamma$ from $\pi^0$ flight dir., in $e^+e^-$ rest frame [deg]")
axes[0].set_ylabel("Dalitz decays")
axes[0].set_title(r"$\gamma$ collinear with $\pi^0$ by construction (~0)")

axes[1].hist(np.cos(np.radians(ep_angle)), bins=np.linspace(-1, 1, 51), histtype="step", color="tab:red")
axes[1].set_xlabel(r"$\cos$(angle of $e^+$ from $\pi^0$ flight dir.), in $e^+e^-$ rest frame")
axes[1].set_ylabel("Dalitz decays")
axes[1].set_title("Dalitz lepton angle")
fig.tight_layout()
savefig("gamma_and_lepton_angle_ee_frame")
plt.show()